<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/18A_GES_Aware_Genomic_RAG_Cell_7C11R_A004_Deterministic_Input_Completeness_Remediation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm Google Drive is mounted and the project directory is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, frozen lineage, and fail-closed output paths

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import hashlib
import json
import re
import tempfile

import pandas as pd
import pyarrow.parquet as pq


NOTEBOOK_NAME = (
    '18A_GES_Aware_Genomic_RAG_Cell_7C11R_'
    'A004_Deterministic_Input_Completeness_Remediation.ipynb'
)
CELL_ID = '7C11R'
STAGE = '7C'
AMENDMENT_ID = 'A004'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()
EXPECTED_RESPONSES = 1_440

EXPECTED_CELL_7C11_TERMINAL_DECISION = (
    'PASS_STAGE7C11_PROTOCOL_AMENDMENT_A004_FULLY_AUTOMATED_STRUCTURED_EVALUATION_'
    'FROZEN_BEFORE_SCORING_A003_HUMAN_PATH_SUPERSEDED_PRIMARY_ENDPOINT_CHANGED_TO_'
    'AUTOMATED_EVIDENCE_FIDELITY_COMPOSITE_FOUR_COMPONENTS_DVSA_PRIMARY_DVSBCEF_'
    'SECONDARY_2000_PAIRED_QUESTION_BOOTSTRAP_LATER_CELL7C12_BLINDED_RESPONSE_LEVEL_'
    'SCORING_ONLY_AUTHORIZED_NO_HUMAN_REVIEW_CONDITION_UNBLINDING_ROUTING_ACCESS_'
    'RUN_AGGREGATION_BOOTSTRAP_OR_ARM_COMPARISON'
)

EXPECTED_CELL_7C11_AUTHORIZATION_DECISION = (
    'AUTHORIZE_STAGE7C_CELL7C12_FULLY_AUTOMATED_BLINDED_STRUCTURED_EVALUATION_'
    'OF_1440_FROZEN_RESPONSES_USING_A004_PRIMARY_AUTOMATED_EVIDENCE_FIDELITY_'
    'COMPOSITE_AND_PRESPECIFIED_SECONDARY_ENDPOINTS_FROM_CELL7C10_DETERMINISTIC_'
    'INPUT_ONLY_NO_HUMAN_REVIEW_CONDITION_UNBLINDING_INTERNAL_ROUTING_MAP_ACCESS_'
    'RUN_AGGREGATION_BOOTSTRAP_OR_ARM_COMPARISON'
)

CELL_7C11_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'protocol_amendment_A004_fully_automated_structured_evaluation_v1'
)
CELL_7C11_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'protocol_amendment_A004_fully_automated_structured_evaluation_v1'
)

CELL_7C11 = OrderedDict([
    ('amendment', {
        'path': CELL_7C11_DIR / 'protocol_amendment_A004_fully_automated_structured_evaluation_v1.json',
        'sha256': '29473b1d69f68cc0b2d49c9f89fe1187825d0b09e40ed7b378ca91d48bc6a063',
    }),
    ('endpoint_spec', {
        'path': CELL_7C11_DIR / 'protocol_amendment_A004_automated_endpoint_spec_v1.json',
        'sha256': '40cf8c54be9aa0337240caa6f6c1f8a31d0baf3078c94638fe8bf79e212edb97',
    }),
    ('aggregation_inference_spec', {
        'path': CELL_7C11_DIR / 'protocol_amendment_A004_aggregation_and_inference_spec_v1.json',
        'sha256': 'b4ea6ae437edfbdc626d249855203c2b6f9b2b6131212d48424760cdeaf6d1f9',
    }),
    ('input_inventory', {
        'path': CELL_7C11_DIR / 'protocol_amendment_A004_verified_input_inventory_v1.csv',
        'sha256': 'e79bb49ec24a8a9a26554b1485338a145ce9e0f8e6a6babaa647ba880447fd7a',
    }),
    ('qc', {
        'path': CELL_7C11_QC_DIR / 'protocol_amendment_A004_qc_v1.json',
        'sha256': '16cda899113d1f8ff5cb4461d71aafc533ceaec3f2a9b4f35c2096950f102f23',
    }),
    ('manifest', {
        'path': CELL_7C11_DIR / 'protocol_amendment_A004_manifest_v1.json',
        'sha256': 'bf6d8eaa2c14e6f0ccce90be3aef2d8c40540453c46e4bfe657c6fd6e567ce13',
    }),
])

CELL_7C10_EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c10_hybrid_single_reviewer_evaluation_packet_v1'
)

CELL_7C10_DETERMINISTIC_INPUT = {
    'path': CELL_7C10_EXEC_DIR / 'cell_7c10_deterministic_scoring_input_v1.parquet',
    'sha256': '7b987c894bf82e4607ba49f270b14d68d9d4ba201e86234a02e225fd279ceb5f',
}
CELL_7C10_FIRSTPASS_PACKET = {
    'path': CELL_7C10_EXEC_DIR / 'cell_7c10_firstpass_single_reviewer_packet_v1.parquet',
    'sha256': 'e1c323cc491c133ea367ee47934ea32f95551053fa3501cc7c5169022d1c0b14',
}

OUT_DIR = (
    ROOT / 'data_processed' / 'stage7_rag'
    / 'cell_7c11r_a004_deterministic_input_remediation_v1'
)
CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c11r_a004_deterministic_input_remediation_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c11r_a004_deterministic_input_remediation_v1'
)

OUTPUTS = OrderedDict([
    ('corrected_deterministic_input',
     OUT_DIR / 'cell_7c11r_a004_deterministic_scoring_input_v2.parquet'),
    ('remediation_authorization',
     CONFIG_DIR / 'cell_7c11r_a004_input_completeness_remediation_authorization_v1.json'),
    ('input_inventory',
     CONFIG_DIR / 'cell_7c11r_verified_input_inventory_v1.csv'),
    ('qc',
     QC_DIR / 'cell_7c11r_a004_input_completeness_remediation_qc_v1.json'),
    ('manifest',
     CONFIG_DIR / 'cell_7c11r_a004_input_completeness_remediation_manifest_v1.json'),
])

for directory in (OUT_DIR, CONFIG_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing = [str(path) for path in OUTPUTS.values() if path.exists()]
if existing:
    raise FileExistsError(
        'Cell 7C11R fail-closed overwrite protection is active. Existing output(s):\\n- '
        + '\\n- '.join(existing)
    )

print(f'Output directory: {OUT_DIR}')
print(f'Config directory: {CONFIG_DIR}')
print(f'QC directory    : {QC_DIR}')

Output directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage7_rag/cell_7c11r_a004_deterministic_input_remediation_v1
Config directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c11r_a004_deterministic_input_remediation_v1
QC directory    : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c11r_a004_deterministic_input_remediation_v1


## 2. Checksum and stable-write helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(f'Invalid SHA-256 sidecar: {path}')
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    return (
        path.exists()
        and sidecar_path(path).exists()
        and read_sidecar_hash(sidecar_path(path)) == sha256_file(path)
    )


def verify_exact_artifact(label: str, path: Path, expected_sha256: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'{label} SHA-256 mismatch.\\nExpected: {expected_sha256}\\nObserved: {observed}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def stable_write_json(path: Path, payload: Any) -> str:
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        ) + chr(10),
        encoding='utf-8',
    )
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator=chr(10))
    return sha256_file(path)


def stable_write_parquet(path: Path, frame: pd.DataFrame) -> str:
    frame.to_parquet(path, index=False, engine='pyarrow', compression='zstd')
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    sidecar_path(path).write_text(
        f'{sha256_file(path)}  {path.name}' + chr(10),
        encoding='utf-8',
    )


with tempfile.TemporaryDirectory(prefix='cell_7c11r_writer_test_') as tmp:
    p = Path(tmp) / 'x.json'
    stable_write_json(p, {'ok': True})
    write_sidecar(p)
    assert load_json(p) == {'ok': True}
    assert sidecar_is_valid(p)

print('Serialization / SHA-256 helper self-test: PASS')

Serialization / SHA-256 helper self-test: PASS


## 3. Reverify A004 and confirm the input-completeness defect before writing anything

In [4]:
verified_inputs = []

for artifact_id, spec in CELL_7C11.items():
    record = verify_exact_artifact(
        f'cell_7c11_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7C11'
    verified_inputs.append(record)

for label, spec in [
    ('cell_7c10_deterministic_scoring_input', CELL_7C10_DETERMINISTIC_INPUT),
    ('cell_7c10_firstpass_review_packet', CELL_7C10_FIRSTPASS_PACKET),
]:
    record = verify_exact_artifact(label, spec['path'], spec['sha256'])
    record['source_cell'] = '7C10'
    verified_inputs.append(record)

amendment_7c11 = load_json(CELL_7C11['amendment']['path'])
endpoint_spec = load_json(CELL_7C11['endpoint_spec']['path'])
manifest_7c11 = load_json(CELL_7C11['manifest']['path'])
qc_7c11 = load_json(CELL_7C11['qc']['path'])

if manifest_7c11.get('terminal_decision') != EXPECTED_CELL_7C11_TERMINAL_DECISION:
    raise AssertionError('Cell 7C11 terminal PASS mismatch.')
if amendment_7c11.get('authorization_decision') != EXPECTED_CELL_7C11_AUTHORIZATION_DECISION:
    raise AssertionError('Cell 7C11 Cell 7C12 authorization mismatch.')
if int(qc_7c11.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C11 QC does not report zero failures.')
if manifest_7c11.get('condition_unblinding_authorized') is not False:
    raise AssertionError('Cell 7C11 unexpectedly authorized unblinding.')

deterministic_input = pd.read_parquet(CELL_7C10_DETERMINISTIC_INPUT['path'])
firstpass_packet = pd.read_parquet(
    CELL_7C10_FIRSTPASS_PACKET['path'],
    columns=['review_item_id', 'question_id', 'structured_answer_key_json'],
)

if len(deterministic_input) != EXPECTED_RESPONSES:
    raise AssertionError('Cell 7C10 deterministic input must contain 1,440 rows.')
if len(firstpass_packet) != EXPECTED_RESPONSES:
    raise AssertionError('Cell 7C10 first-pass packet must contain 1,440 rows.')

if deterministic_input['review_item_id'].duplicated().any():
    raise AssertionError('Duplicate review_item_id in deterministic input.')
if firstpass_packet['review_item_id'].duplicated().any():
    raise AssertionError('Duplicate review_item_id in first-pass packet.')

if 'aggregate_conflict_flag' in deterministic_input.columns:
    raise AssertionError(
        'The expected input-completeness defect is absent: aggregate_conflict_flag is already present. '
        'Do not run this remediation against a different input version.'
    )

primary_components = endpoint_spec['primary_automated_endpoint']['components']
if 'conflict_concordance_pass' not in primary_components:
    raise AssertionError('Frozen A004 endpoint no longer includes conflict_concordance_pass.')

print('Cell 7C11 / A004 package              : 6/6 exact hashes + sidecars')
print('Cell 7C10 required inputs              : 2/2 exact hashes + sidecars')
print('A004 terminal PASS                     : VERIFIED')
print('Frozen primary endpoint components      : 4')
print('aggregate_conflict_flag in V1 input     : NO — defect confirmed')
print('Condition identity opened               : NO')
print('Scoring performed                       : NO')

Cell 7C11 / A004 package              : 6/6 exact hashes + sidecars
Cell 7C10 required inputs              : 2/2 exact hashes + sidecars
A004 terminal PASS                     : VERIFIED
Frozen primary endpoint components      : 4
aggregate_conflict_flag in V1 input     : NO — defect confirmed
Condition identity opened               : NO
Scoring performed                       : NO


## 4. Extract only the frozen answer-key conflict target and build corrected deterministic input V2

In [5]:
def normalize_bool(value: Any, *, field_name: str, review_item_id: str) -> bool:
    if isinstance(value, bool):
        return value
    if value in (0, 1):
        return bool(value)
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {'true', '1'}:
            return True
        if normalized in {'false', '0'}:
            return False
    raise ValueError(
        f'Invalid Boolean value for {field_name} at {review_item_id}: {value!r}'
    )


target_rows = []

for row in firstpass_packet.itertuples(index=False):
    answer_key = json.loads(row.structured_answer_key_json)

    if 'aggregate_conflict_flag' not in answer_key:
        raise KeyError(
            f'aggregate_conflict_flag missing from structured answer key for {row.review_item_id}'
        )

    target_rows.append({
        'review_item_id': str(row.review_item_id),
        'question_id': str(row.question_id),
        'expected_aggregate_conflict_flag': normalize_bool(
            answer_key['aggregate_conflict_flag'],
            field_name='aggregate_conflict_flag',
            review_item_id=str(row.review_item_id),
        ),
    })

conflict_targets = pd.DataFrame(target_rows)

if len(conflict_targets) != 1_440:
    raise AssertionError('Conflict-target table must contain 1,440 rows.')
if conflict_targets['review_item_id'].duplicated().any():
    raise AssertionError('Duplicate review_item_id in conflict-target table.')

corrected_input = deterministic_input.merge(
    conflict_targets,
    on=['review_item_id', 'question_id'],
    how='left',
    validate='one_to_one',
)

if len(corrected_input) != 1_440:
    raise AssertionError('Corrected deterministic input must contain 1,440 rows.')
if corrected_input['expected_aggregate_conflict_flag'].isna().any():
    raise AssertionError('Missing expected_aggregate_conflict_flag after one-to-one join.')

# Preserve all original rows and columns in original order, append exactly one field.
original_columns = list(deterministic_input.columns)
expected_columns = original_columns + ['expected_aggregate_conflict_flag']
if list(corrected_input.columns) != expected_columns:
    corrected_input = corrected_input[expected_columns]

for column in original_columns:
    left = deterministic_input[column].reset_index(drop=True)
    right = corrected_input[column].reset_index(drop=True)
    if not left.equals(right):
        raise AssertionError(
            f'Original deterministic-input column changed during remediation: {column}'
        )

print(f'Corrected deterministic input rows      : {len(corrected_input):,}')
print(f'Original columns preserved              : {len(original_columns)} / {len(original_columns)}')
print('New field                               : expected_aggregate_conflict_flag')
print('Scientific endpoint changed             : NO')
print('Condition identity required             : NO')

Corrected deterministic input rows      : 1,440
Original columns preserved              : 10 / 10
New field                               : expected_aggregate_conflict_flag
Scientific endpoint changed             : NO
Condition identity required             : NO


## 5. Freeze corrected input and authorize Cell 7C12 scoring from V2 only

In [6]:
authorization_decision = (
    'AUTHORIZE_STAGE7C_CELL7C12_FULLY_AUTOMATED_BLINDED_STRUCTURED_EVALUATION_'
    'OF_1440_FROZEN_RESPONSES_USING_A004_ENDPOINT_SPEC_AND_CELL7C11R_CORRECTED_'
    'DETERMINISTIC_SCORING_INPUT_V2_WITH_EXPECTED_AGGREGATE_CONFLICT_FLAG_NO_'
    'HUMAN_REVIEW_CONDITION_UNBLINDING_INTERNAL_ROUTING_ACCESS_RUN_AGGREGATION_'
    'BOOTSTRAP_OR_ARM_COMPARISON'
)

prewrite_checks = OrderedDict([
    ('cell7c11_6_artifacts_verified', len([x for x in verified_inputs if x['source_cell'] == '7C11']) == 6),
    ('cell7c10_2_inputs_verified', len([x for x in verified_inputs if x['source_cell'] == '7C10']) == 2),
    ('a004_terminal_pass_exact',
     manifest_7c11.get('terminal_decision') == EXPECTED_CELL_7C11_TERMINAL_DECISION),
    ('v1_input_1440', len(deterministic_input) == 1440),
    ('firstpass_packet_1440', len(firstpass_packet) == 1440),
    ('v1_missing_conflict_target_confirmed',
     'aggregate_conflict_flag' not in deterministic_input.columns),
    ('answer_key_conflict_target_available',
     len(conflict_targets) == 1440),
    ('corrected_input_1440', len(corrected_input) == 1440),
    ('corrected_unique_review_items', corrected_input['review_item_id'].nunique() == 1440),
    ('conflict_target_complete',
     corrected_input['expected_aggregate_conflict_flag'].notna().all()),
    ('only_one_new_field',
     len(corrected_input.columns) == len(deterministic_input.columns) + 1),
    ('new_field_exact_name',
     corrected_input.columns[-1] == 'expected_aggregate_conflict_flag'),
    ('condition_unblinding_not_performed', True),
    ('routing_map_not_opened', True),
    ('run_aggregation_not_performed', True),
    ('bootstrap_not_performed', True),
    ('arm_comparison_not_performed', True),
    ('response_scoring_not_performed', True),
])

failed = [name for name, passed in prewrite_checks.items() if not bool(passed)]
if failed:
    raise RuntimeError(
        'Cell 7C11R prewrite QC failed:\\n- ' + '\\n- '.join(failed)
    )

stable_write_parquet(OUTPUTS['corrected_deterministic_input'], corrected_input)
write_sidecar(OUTPUTS['corrected_deterministic_input'])

remediation_authorization = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'version': '1.0.0',
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'defect':
        'Cell 7C10 deterministic-scoring input lacked the frozen answer-key aggregate_conflict_flag '
        'required to calculate A004 conflict_concordance_pass.',
    'remediation':
        'Append expected_aggregate_conflict_flag extracted from the frozen structured_answer_key_json '
        'in the score-blind Cell 7C10 first-pass packet. Preserve every original deterministic-input '
        'column and row unchanged.',
    'scientific_endpoint_changed': False,
    'A004_endpoint_spec_sha256': CELL_7C11['endpoint_spec']['sha256'],
    'source_deterministic_input_sha256': CELL_7C10_DETERMINISTIC_INPUT['sha256'],
    'source_firstpass_packet_sha256': CELL_7C10_FIRSTPASS_PACKET['sha256'],
    'corrected_deterministic_input_sha256':
        sha256_file(OUTPUTS['corrected_deterministic_input']),
    'authorization_decision': authorization_decision,
    'next_authorized_cell': '7C12',
    'cell_7c12_allowed_scoring_input':
        str(OUTPUTS['corrected_deterministic_input']),
    'human_review_authorized': False,
    'condition_unblinding_authorized': False,
    'internal_routing_map_access_authorized': False,
    'run_aggregation_authorized': False,
    'bootstrap_inference_authorized': False,
    'arm_comparison_authorized': False,
}
stable_write_json(OUTPUTS['remediation_authorization'], remediation_authorization)
write_sidecar(OUTPUTS['remediation_authorization'])

input_inventory = pd.DataFrame(verified_inputs)
stable_write_csv(OUTPUTS['input_inventory'], input_inventory)
write_sidecar(OUTPUTS['input_inventory'])

terminal_decision = (
    'PASS_STAGE7C11R_A004_INPUT_COMPLETENESS_REMEDIATED_BEFORE_SCORING_'
    'CELL7C10_DETERMINISTIC_INPUT_V1_PRESERVED_CORRECTED_V2_1440_ROWS_WITH_'
    'EXPECTED_AGGREGATE_CONFLICT_FLAG_APPENDED_FROM_FROZEN_SCORE_BLIND_ANSWER_'
    'KEYS_A004_ENDPOINT_UNCHANGED_CELL7C12_BLINDED_AUTOMATED_SCORING_ONLY_'
    'AUTHORIZED_NO_HUMAN_REVIEW_CONDITION_UNBLINDING_ROUTING_ACCESS_RUN_'
    'AGGREGATION_BOOTSTRAP_OR_ARM_COMPARISON'
)

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'checks': {name: bool(value) for name, value in prewrite_checks.items()},
    'passed_checks': len(prewrite_checks),
    'failed_checks': 0,
    'total_checks': len(prewrite_checks),
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'upstream_lineage': {
        'cell_7c11_manifest_sha256': CELL_7C11['manifest']['sha256'],
        'cell_7c11_endpoint_spec_sha256': CELL_7C11['endpoint_spec']['sha256'],
        'cell_7c10_deterministic_input_sha256': CELL_7C10_DETERMINISTIC_INPUT['sha256'],
        'cell_7c10_firstpass_packet_sha256': CELL_7C10_FIRSTPASS_PACKET['sha256'],
    },
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_valid': sidecar_is_valid(path),
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
    'authorization_decision': authorization_decision,
    'terminal_decision': terminal_decision,
    'next_authorized_cell': '7C12',
    'condition_unblinding_authorized': False,
    'internal_routing_map_access_authorized': False,
    'run_aggregation_authorized': False,
    'bootstrap_inference_authorized': False,
    'arm_comparison_authorized': False,
}
stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

# Final fresh readback.
for path in OUTPUTS.values():
    if not path.exists() or not sidecar_is_valid(path):
        raise AssertionError(f'Cell 7C11R final readback failed: {path}')

rb = pd.read_parquet(OUTPUTS['corrected_deterministic_input'])
rb_auth = load_json(OUTPUTS['remediation_authorization'])
rb_qc = load_json(OUTPUTS['qc'])
rb_manifest = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('readback_1440', len(rb) == 1440),
    ('readback_unique_1440', rb['review_item_id'].nunique() == 1440),
    ('readback_conflict_target_present',
     'expected_aggregate_conflict_flag' in rb.columns),
    ('readback_conflict_target_complete',
     rb['expected_aggregate_conflict_flag'].notna().all()),
    ('endpoint_unchanged',
     rb_auth['scientific_endpoint_changed'] is False),
    ('next_cell_7c12', rb_manifest.get('next_authorized_cell') == '7C12'),
    ('unblinding_false',
     rb_manifest.get('condition_unblinding_authorized') is False),
    ('routing_false',
     rb_manifest.get('internal_routing_map_access_authorized') is False),
    ('aggregation_false',
     rb_manifest.get('run_aggregation_authorized') is False),
    ('bootstrap_false',
     rb_manifest.get('bootstrap_inference_authorized') is False),
    ('arm_comparison_false',
     rb_manifest.get('arm_comparison_authorized') is False),
    ('qc_zero_failures', int(rb_qc.get('failed_checks', -1)) == 0),
    ('all_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_rb = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_rb:
    raise RuntimeError(
        'Cell 7C11R final readback QC failed:\\n- ' + '\\n- '.join(failed_rb)
    )

total_checks = len(prewrite_checks) + len(readback_checks)

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C11R')
print('A004 DETERMINISTIC-INPUT COMPLETENESS REMEDIATION — BEFORE SCORING')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nUPSTREAM REVERIFICATION')
print(f'Cell 7C11 manifest SHA-256                    : {CELL_7C11["manifest"]["sha256"]}')
print('Cell 7C11 terminal PASS verified              : YES')
print('Cell 7C11 A004 endpoint                       : UNCHANGED')
print('Cell 7C10 deterministic input                 : 1,440 rows — exact SHA verified')
print('Cell 7C10 first-pass packet                   : 1,440 rows — exact SHA verified')

print('\\nINPUT-COMPLETENESS DEFECT')
print('Response conflict_detected                    : PRESENT in Cell 7C10 V1 input')
print('Expected aggregate conflict flag              : ABSENT in Cell 7C10 V1 input')
print('A004 conflict_concordance calculable from V1  : NO')

print('\\nREMEDIATION')
print('Corrected deterministic input V2              : 1,440 rows')
print('Original deterministic columns changed        : NO')
print('New field appended                            : expected_aggregate_conflict_flag')
print('Source                                         : frozen structured answer keys')
print('Condition identity opened                     : NO')
print('Internal routing map opened                   : NO')
print('Response scoring performed                    : NO')

print('\\nCELL 7C11R FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {total_checks}/{total_checks} PASS')

print('\\nNEXT AUTHORIZED CELL')
print('Stage 7C — Cell 7C12                          : blinded automated response-level scoring + freeze')
print('Allowed scoring input                         : Cell 7C11R corrected deterministic input V2 only')
print('Condition unblinding / routing access         : PROHIBITED')
print('Run aggregation / bootstrap / arm comparison  : PROHIBITED')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C11R
A004 DETERMINISTIC-INPUT COMPLETENESS REMEDIATION — BEFORE SCORING
Notebook                                      : 18A_GES_Aware_Genomic_RAG_Cell_7C11R_A004_Deterministic_Input_Completeness_Remediation.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nUPSTREAM REVERIFICATION
Cell 7C11 manifest SHA-256                    : bf6d8eaa2c14e6f0ccce90be3aef2d8c40540453c46e4bfe657c6fd6e567ce13
Cell 7C11 terminal PASS verified              : YES
Cell 7C11 A004 endpoint                       : UNCHANGED
Cell 7C10 deterministic input                 : 1,440 rows — exact SHA verified
Cell 7C10 first-pass packet                   : 1,440 rows — exact SHA verified
\nINPUT-COMPLETENESS DEFECT
Response conflict_detected                    : PRESENT in Cell